# Kalshi Bot v2 — full pipeline

Thin notebook. All logic lives in `kalshi_v2/`. Cells here just
drive the lifecycle (build clients → start bot → inspect → stop)
and print results inline.

**Prereq.** `~/.kalshi/credentials.env` must contain:
```
KALSHI_PROD_KEY_ID=...
KALSHI_PROD_PRIVATE_KEY_PATH=~/.kalshi/prod_private_key.pem
```

**Pipeline.** Coinbase BTC spot → Kalshi WS orderbooks → empirical
bank fair value (drift-removed, vol+kurt matched) → HRDNN P-robust
filter (16 bootstrapped measures) → Lipschitz-clamped sizing →
risk preflight → place / record.

In [9]:
# Install / upgrade websockets in this kernel (v13+ uses
# additional_headers; <=10.x uses extra_headers — kalshi_v2/data.py
# auto-detects whichever is available, but v13+ is preferred).
%pip install -q 'websockets>=13' cryptography requests pandas numpy scipy python-dateutil
print('deps ok — RESTART KERNEL if websockets was upgraded, then re-run from cell 1')

Note: you may need to restart the kernel to use updated packages.
deps ok — RESTART KERNEL if websockets was upgraded, then re-run from cell 1


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import time
from datetime import datetime, timezone

from kalshi_v2.config import CFG
from kalshi_v2.client import KalshiClient
from kalshi_v2 import data as v2data
from kalshi_v2.data import (fetch_historical_minutes, add_rv_features,
                              SPOT, BOOKS, TRACKED, WS_STATE, BOT_STATE)
from kalshi_v2.model import build_empirical_bank
from kalshi_v2.robust import build_ambiguity_set, SIZER
from kalshi_v2.strategy import scan_signals
from kalshi_v2.paper_db import open_trades, settled_trades
from kalshi_v2.risk import RISK_BLOCKS, get_live_balance
from kalshi_v2.portfolio import (paper_portfolio_metrics,
                                   live_portfolio_metrics, portfolio_metrics)
from kalshi_v2.main import (start_bot, stop_bot, status, kill_switch,
                              enable_live, disable_live, cancel_all_live_orders,
                              dashboard, tail_log)

print('imports OK')
print(f'mode:          {CFG["mode"]}')
print(f'live_enabled:  {CFG["live_enabled"]}')
print(f'robust_enabled: {CFG["robust_enabled"]}')
print(f'sfm_enabled:   {CFG["sfm_enabled"]}')

imports OK
mode:          paper
live_enabled:  False
robust_enabled: True
sfm_enabled:   False


## 2. Build clients

Two clients: `kalshi_md` for read-only market data (no auth required
for public endpoints) and `kalshi_live` for authed actions (orders,
balance, WS handshake).

In [2]:
kalshi_md   = KalshiClient(env='prod')
kalshi_live = KalshiClient(env='prod')

print(f'md client:    base={kalshi_md.base_url}, signed={kalshi_md.private_key is not None}')
print(f'live client:  base={kalshi_live.base_url}, signed={kalshi_live.private_key is not None}')
print(f'live key_id:  {kalshi_live.key_id[:8] + "..." if kalshi_live.key_id else None}')

if kalshi_live.private_key is None:
    print('\n  ⚠ live client unsigned — bot will run paper-only, no WS auth')

md client:    base=https://api.elections.kalshi.com/trade-api/v2, signed=True
live client:  base=https://api.elections.kalshi.com/trade-api/v2, signed=True
live key_id:  628659b8...


In [3]:
# Quick sanity check: hit a public endpoint
try:
    ev = kalshi_md.get_events(series_ticker='KXBTC', status='open', limit=3)
    print(f'public REST OK — {len(ev.get("events", []))} BTC events open')
    for e in ev.get('events', [])[:3]:
        print(f'  {e.get("event_ticker")}')
except Exception as e:
    print(f'REST sanity failed: {e}')

if kalshi_live.private_key is not None:
    try:
        bal = kalshi_live.get_balance()
        print(f'\nlive balance: ${float(bal.get("balance", 0))/100:.2f}')
    except Exception as e:
        print(f'\nbalance fetch failed: {e}')

public REST OK — 3 BTC events open
  KXBTC-26MAY1517
  KXBTC-26MAY1317
  KXBTC-26MAY1312

live balance: $103.07


In [4]:
from kalshi_v2.config import CFG
for k in ['min_edge_cents', 'max_spread_cents', 'min_entry_price', 'max_entry_price',
          'brti_dampening', 'empirical_blend', 'market_shrink',
          'no_side_edge_surcharge_cents']:
    print(f'  {k:30s} {CFG[k]}')
# min_yes_p / max_no_p should raise KeyError (they're gone)

  min_edge_cents                 2.5
  max_spread_cents               3
  min_entry_price                0.2
  max_entry_price                0.8
  brti_dampening                 0.8
  empirical_blend                0.7
  market_shrink                  0.1
  no_side_edge_surcharge_cents   3.0


## 3. Show config

In [5]:
for k in sorted(CFG.keys()):
    v = CFG[k]
    print(f'  {k:30s} {v}')

  arb_max_dollars_per_trade      1000
  bankroll                       100000.0
  brti_dampening                 0.8
  db_path                        /Users/rithvikijju/.btc_kalshi_bot/v2.db
  decision_interval_sec          5.0
  empirical_blend                0.7
  event_series                   ('KXBTC', 'KXBTCD')
  i_acknowledge_real_money_risk  False
  kalshi_fee_cap                 0.07
  kelly_multiplier               0.5
  lipschitz_position_L           200
  live_enabled                   False
  market_shrink                  0.1
  max_concurrent_signals         1
  max_entry_price                0.8
  max_per_market                 0.02
  max_position_age_min           90
  max_spread_cents               3
  min_edge_cents                 2.5
  min_entry_price                0.2
  min_liquidity                  0
  min_model_confidence           0.0
  mode                           paper
  no_near_strike_distance_usd    115
  no_near_strike_max_contracts   1
  no_side_edge_su

## 4. Fetch BTC history + features

90 days of BTC 1-min bars from Coinbase. This populates the input to
`build_empirical_bank` and `build_ambiguity_set`. Takes 30–60 s.

In [6]:
btc_1m = add_rv_features(fetch_historical_minutes(days_back=90))
print(f'btc_1m: {len(btc_1m):,} bars')
print(f'  range:   {btc_1m["time"].min()}  →  {btc_1m["time"].max()}')
print(f'  spot:    ${btc_1m["close"].iloc[-1]:,.0f}')
print(f'  rv_60m last:  {btc_1m["rv_60m"].iloc[-1]:.4f} (annualized)')
btc_1m.tail(3)

btc_1m: 129,160 bars
  range:   2026-02-12 15:06:00+00:00  →  2026-05-13 15:05:00+00:00
  spot:    $79,632
  rv_60m last:  0.3029 (annualized)


,time,low,high,open,close,volume,log_ret,rv_5m,rv_15m,rv_60m,rv_240m,rv_1440m
129157,2026-05-13 15:03:00+00:00,79767.26,79793.75,79773.28,79767.26,3.715205,-0.000075,0.176964,0.297005,0.293608,0.366161,0.267665
129158,2026-05-13 15:04:00+00:00,79697.80,79769.99,79764.25,79702.00,9.246302,-0.000818,0.357136,0.345049,0.304261,0.367773,0.268114
129159,2026-05-13 15:05:00+00:00,79627.19,79702.00,79701.99,79632.40,11.755927,-0.000874,0.344003,0.374804,0.302866,0.368918,0.268612


## 5. Build empirical bank — preview before bot starts

Diagnostic: same call `start_bot` will make. Confirms drift removal
and conditioning features are sane.

In [7]:
bank = build_empirical_bank(btc_1m, horizon_min=60, n_samples=5000, demean=True)
R = bank['log_returns']
import numpy as np
print(f'  n samples:     {bank["n"]:,}')
print(f'  R mean (post-demean): {R.mean():+.6f}')
print(f'  R std:         {R.std():.6f}')
print(f'  R quantiles:   1%={np.quantile(R, 0.01):+.4f}, 50%={np.quantile(R, 0.50):+.4f}, 99%={np.quantile(R, 0.99):+.4f}')
print(f'  v_mean:        {bank["v_mean"]:.4f}')
print(f'  k_mean:        {bank["k_mean"]:+.3f}')

  n samples:     5,000
  R mean (post-demean): +0.000000
  R std:         0.004601
  R quantiles:   1%=-0.0126, 50%=-0.0001, 99%=+0.0139
  v_mean:        0.4031
  k_mean:        +2.171


## 6. Build ambiguity set — preview

16 bootstrapped empirical banks. The robust filter rejects a signal
unless **every** measure agrees the trade has positive post-fee edge.
Spread of v_mean across measures shows the filter has real ambiguity
to test against (vs a near-zero spread, which would mean it's a noop).

In [8]:
amb = build_ambiguity_set(btc_1m, horizon_min=60,
                            n_bootstrap=CFG['robust_n_bootstrap'])
v_means = [m['v_mean'] for m in amb]
k_means = [m['k_mean'] for m in amb]
print(f'  measures:        {len(amb)}')
print(f'  v_mean range:   [{min(v_means):.4f}, {max(v_means):.4f}]  (spread {max(v_means)-min(v_means):.4f})')
print(f'  k_mean range:   [{min(k_means):+.3f}, {max(k_means):+.3f}]  (spread {max(k_means)-min(k_means):.3f})')
if max(v_means) - min(v_means) < 0.01:
    print('  ⚠ low spread — bootstrap may not be giving ambiguity (degenerate ambiguity set)')
else:
    print('  ✓ measures vary — filter has something to test against')

  measures:        16
  v_mean range:   [0.3909, 0.4290]  (spread 0.0381)
  k_mean range:   [+2.045, +2.378]  (spread 0.333)
  ✓ measures vary — filter has something to test against


## 7. Start the bot

Spins up 4 daemon threads:
- `spot_poller` — Coinbase BTC spot every 2 s
- `ws_listener` — Kalshi WS orderbook stream (REST fallback on 401/403)
- `event_tracker` — finds nearest BTC event in TTL window every 60 s
- `decision`     — settle → manage → scan → execute every `decision_interval_sec`

Idempotent: re-running `start_bot` while running prints a warning and
no-ops. Pass `btc_1m=btc_1m, refresh_btc_1m=False` to skip the 90-day
refetch (we already have it).

In [9]:
start_bot(kalshi_md, kalshi_live, btc_1m=btc_1m, refresh_btc_1m=False)

  ✓ empirical bank: 5000 samples
  ✓ ambiguity set: 16 measures
  ✓ live balance: $103.07

✓ bot running (4 threads). mode=paper


## 8. Live status

Re-run this cell any time to see thread health, WS state, current
tracked event, recent log lines.

In [14]:
status(last_n_log_lines=20)

  V2 BOT STATUS @ 2026-05-13 10:10:05 CDT
  mode:          paper
  live_enabled:  False
  running:       True
  iter:          2
  trades:        0
  live balance:  $103.07

  Threads: 4
    v2_spot_poller             alive=True
    v2_ws_listener             alive=True
    v2_event_tracker           alive=True
    v2_decision                alive=True

  WebSocket:
    mode:          websocket
    connected:     True
    subscribed:    KXBTC-26MAY1312
    msgs received: 6
    last msg:      10:10:05 CDT

  Tracked event:  KXBTC-26MAY1312
  Books in mem:   188

  Empirical bank: 5000
  Ambiguity set:  16 measures

  Log tail:
    [10:09:57 CDT] WS connected
    [10:09:57 CDT] tracking: KXBTC-26MAY1312 closes 2026-05-13 16:00:00+00:00
    [10:09:57 CDT] REST seed: 188 markets for KXBTC-26MAY1312
    [10:10:02 CDT] REST seed: 188 markets for KXBTC-26MAY1312
    [10:10:03 CDT] WS resubscribed: 188 tickers
    [10:10:03 CDT] WS sub confirmed sid=1
    [10:10:03 CDT] WS sub confirmed sid=2


## 8b. Full live dashboard

Re-runnable snapshot of the entire pipeline. Shows:
- threads + WS state
- spot, sigma, tracked event, sample books
- per-market edge breakdown for every market in scope, with the reason each one was rejected (no quote, spread too wide, edge too thin, entry out of band, robust filter rejection, etc.)
- last 10 robust filter decisions
- last 10 risk-preflight blocks
- open + settled trades + realized PnL
- recent log tail

## 8c. Stream the log

`tail_log(30)` prints the last 30 lines.
`tail_log(30, follow_secs=60)` blocks and streams new entries for 60s.

In [15]:
tail_log(30)
# Or stream live for a minute:
# tail_log(30, follow_secs=60)

[18:31:17 CDT] WS connected
[18:31:18 CDT] tracking: KXBTC-26MAY1220 closes 2026-05-13 00:00:00+00:00
[18:31:19 CDT] REST seed: 188 markets for KXBTC-26MAY1220
[18:31:22 CDT] REST seed: 188 markets for KXBTC-26MAY1220
[18:31:22 CDT] WS resubscribed: 188 tickers
[18:31:22 CDT] WS sub confirmed sid=1
[18:31:22 CDT] WS sub confirmed sid=2
[18:31:22 CDT] WS sub confirmed sid=3


## 9. Manual signal scan (diagnostic)

Calls the same `scan_signals` the decision worker calls, but inline
so you can see the edge distribution and which markets passed the
robust filter.

In [15]:
from kalshi_v2.main import _EMPIRICAL_BANK, _AMBIGUITY_SET

sigs = scan_signals(empirical_bank=_EMPIRICAL_BANK,
                      ambiguity_set=_AMBIGUITY_SET)
if len(sigs) == 0:
    print('no signals this scan')
    print(f'  spot:     {SPOT.get("price")}')
    print(f'  event:    {TRACKED.get("event")}')
    print(f'  books:    {len(BOOKS)}')
else:
    cols = ['ticker', 'side', 'entry_price', 'model_p_yes', 'edge_c',
            'robust_pass_rate', 'robust_mean_edge_c', 'ttl_min']
    cols = [c for c in cols if c in sigs.columns]
    print(f'{len(sigs)} signal(s) passed all filters:\n')
    print(sigs[cols].to_string(index=False))

no signals this scan
  spot:     79576.575
  event:    KXBTC-26MAY1312
  books:    188


## 10. Open positions

In [16]:
op = open_trades()
if len(op) == 0:
    print('no open positions')
else:
    cols = ['id', 'timestamp_utc', 'market_ticker', 'side', 'contracts',
            'entry_price', 'entry_edge_cents', 'model_p_yes', 'trade_type']
    cols = [c for c in cols if c in op.columns]
    print(f'{len(op)} open positions:\n')
    print(op[cols].to_string(index=False))

no open positions


## 11. Robust filter decisions log

Every signal that came through `scan_signals` (pass or fail) is
logged here. Useful for tuning `robust_min_pass_rate` and
`robust_min_mean_edge_c`.

In [17]:
from kalshi_v2.paper_db import _conn
import pandas as pd
conn = _conn()
rd = pd.read_sql_query(
    'SELECT ts, ticker, side, entry_price, n_measures, pass_rate, '
    'mean_edge_c, min_edge_c, max_edge_c, passed '
    'FROM robust_decisions ORDER BY ts DESC LIMIT 20', conn)
conn.close()
if len(rd) == 0:
    print('no robust decisions logged yet')
else:
    print(rd.to_string(index=False))

                              ts                 ticker side  entry_price  n_measures  pass_rate  mean_edge_c  min_edge_c  max_edge_c  passed
2026-05-13T14:54:38.760810+00:00 KXBTC-26MAY1311-B79650   no         0.78          16        1.0     8.301716    6.792341   10.257341       1
2026-05-13T14:54:01.807784+00:00 KXBTC-26MAY1311-B79850   no         0.75          16        1.0    11.762795   10.515920   13.980920       1
2026-05-13T00:33:39.894825+00:00 KXBTC-26MAY1221-B80650  yes         0.38          16        1.0     6.093943    3.495193    9.585193       1
2026-05-13T00:32:31.763135+00:00 KXBTC-26MAY1221-B80650  yes         0.38          16        1.0     5.491401    2.951713    8.516713       1
2026-05-13T00:32:26.364517+00:00 KXBTC-26MAY1221-B80650  yes         0.38          16        1.0     5.440352    2.926914    8.491914       1
2026-05-13T00:31:03.713315+00:00 KXBTC-26MAY1221-B80650  yes         0.35          16        1.0     8.032350    5.853600   10.998600       1
2026-0

In [59]:
from pathlib import Path
from kalshi_v2.backtest import run_backtest

DUCKDB = Path("/Users/rithvikijju/Downloads/kalshi_ws_hourly_plus_btc15m_20260512/"
              "data/live_capture_gapless/live_capture_gapless_20260512_paused.duckdb")

summary = run_backtest(
    duckdb_path=DUCKDB,
    output_dir=Path("./backtest_outputs/v2_replay"),
    decision_interval_sec=5,   # match live decision cadence
    decision_window_min=30,    # only consider trades in last 30 min before close
)
print(summary)

  v2 BACKTEST  ·  live_capture_gapless_20260512_paused.duckdb
  loading BTC ticks → 1-min bars …
    2,778 bars, 2026-05-06 01:07:00+00:00 → 2026-05-08 04:16:00+00:00
  loading settled outcomes …
    278,893 unique settled markets
  fetched 129,160 external BTC bars for bank
  empirical bank: n=5000
  ambiguity set: 16 measures

  replaying 163 events …
      err: time data "2026-05-11T14:50:12+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 38360. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
      err: time data "2026-05-06T07:29:39+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 12344. You might want to try:
    - passing `format` if your strings have a 

/Users/rithvikijju/edge-bot/kalshi_v2/backtest.py:281: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  post_q["received_at_utc"] = pd.to_datetime(post_q["received_at_utc"], utc=True)


    [20/163] KXBTCD-26MAY1119


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      err: time data "2026-05-06T18:25:03+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 128. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
    [40/163] KXBTCD-26MAY0623
    [60/163] KXBTCD-26MAY1117
    [80/163] KXBTCD-26MAY1017


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    [100/163] KXBTCD-26MAY0819
    [120/163] KXBTCD-26MAY0722
      err: time data "2026-05-09T17:32:17+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 11877. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
    [140/163] KXBTCD-26MAY1212
      err: time data "2026-05-06T17:50:28+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 68574. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alo

In [62]:
from pathlib import Path
from kalshi_v2.backtest import run_backtest
from kalshi_v2.config import CFG

DUCKDB = Path("/Users/rithvikijju/Downloads/kalshi_ws_hourly_plus_btc15m_20260512/"
              "data/live_capture_gapless/live_capture_gapless_20260512_paused.duckdb")

# Override exits to clear structural costs
CFG['take_profit_cents'] = 12.0    # was 5.0 — must beat ~8c structural floor
CFG['stop_loss_pct']     = 0.15    # was 0.20 — tighter to limit losses

summary = run_backtest(duckdb_path=DUCKDB, output_dir=Path("./backtest_outputs/tp12_sl15"))
print(summary)

  v2 BACKTEST  ·  live_capture_gapless_20260512_paused.duckdb
  loading BTC ticks → 1-min bars …
    2,778 bars, 2026-05-06 01:07:00+00:00 → 2026-05-08 04:16:00+00:00
  loading settled outcomes …
    278,893 unique settled markets
  fetched 129,160 external BTC bars for bank
  empirical bank: n=5000
  ambiguity set: 16 measures

  replaying 163 events …
    [20/163] KXBTCD-26MAY0913
      err: time data "2026-05-06T18:25:03+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 128. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
    [40/163] KXBTCD-26MAY0806
    [60/163] KXBTCD-26MAY1003
    [80/163] KXBTCD-26MAY0620
    [100/163] KXBTCD-26MAY1118
      err: time data "2026-05-09T17

/Users/rithvikijju/edge-bot/kalshi_v2/backtest.py:281: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  post_q["received_at_utc"] = pd.to_datetime(post_q["received_at_utc"], utc=True)


    [160/163] KXBTCD-26MAY1009
    [163/163] KXBTCD-26MAY1107

  BACKTEST SUMMARY
  trades:        127
  net PnL:       $-9.83
  gross PnL:     $-5.01
  fees:          $4.82
  win rate:      33.9%  (43W / 84L)
  mean / trade:  $-0.077
  std / trade:   $0.092
  best / worst:  $+0.07 / $-0.24
  throttled:     66

  by side:
       n  wins  net_pnl  mean_pnl  win_rate
side                                       
no    76    25    -6.19    -0.081     0.329
yes   51    18    -3.64    -0.071     0.353

  by strike distance:
              n  wins  net_pnl  mean_pnl  win_rate
dist_bucket                                       
0-50         68    15    -6.32    -0.093     0.221
50-100       29     7    -2.80    -0.097     0.241
100-200      25    17    -0.73    -0.029     0.680
200-400       5     4     0.02     0.004     0.800

  by exit reason:
              n  net_pnl  mean_pnl
exit_kind                         
stop_loss    83   -11.65    -0.140
take_profit  43     1.98     0.046
time_exit   

In [64]:
import pandas as pd
df = pd.read_csv('backtest_outputs/tp12_sl15/trades.csv')
close = df[df['dist_usd'] < 100].copy()
close['market_mid'] = (close['entry_quoted'] if close['side'].iloc[0]=='yes' else 1-close['entry_quoted'])
# Disagreement: did model and market mid disagree on direction?
close['model_thinks_yes'] = close['model_p_yes'] > 0.5
close['market_thinks_yes'] = close['market_mid'] > 0.5
close['disagree'] = close['model_thinks_yes'] != close['market_thinks_yes']

for tag, sub in [('AGREE', close[~close['disagree']]),
                  ('DISAGREE', close[close['disagree']])]:
    if len(sub):
        wr = (sub['net_pnl']>0).mean()
        net = sub['net_pnl'].sum()
        print(f'{tag}:  n={len(sub):2d}  WR={wr*100:.1f}%  net=${net:+.2f}')

AGREE:  n=55  WR=21.8%  net=$-5.22
DISAGREE:  n=42  WR=23.8%  net=$-3.90


In [66]:
from kalshi_v2.config import CFG

CFG['order_buffer_cents']    = 1       # save 2¢ round-trip vs current 2¢
CFG['take_profit_cents']     = 18.0    # need wide TP to clear ~8¢ structural floor
CFG['stop_loss_pct']         = 0.12    # tighten SL to limit downside
CFG['min_edge_cents']        = 6.0     # require larger predicted edge per trade
# Keep robust filter strict
CFG['robust_min_pass_rate']  = 1.0
# Disable awareness layer (didn't help, sometimes hurt)
CFG['brti_dampening']        = 1.0
CFG['empirical_blend']       = 1.0
CFG['market_shrink']         = 0.0
CFG['no_side_edge_surcharge_cents'] = 0.0
CFG['platt_enabled']         = False

summary = run_backtest(duckdb_path=DUCKDB,
                        output_dir=Path("./backtest_outputs/structural_fix"))
print(summary)

  v2 BACKTEST  ·  live_capture_gapless_20260512_paused.duckdb
  loading BTC ticks → 1-min bars …
    2,778 bars, 2026-05-06 01:07:00+00:00 → 2026-05-08 04:16:00+00:00
  loading settled outcomes …
    278,893 unique settled markets
  fetched 129,160 external BTC bars for bank
  empirical bank: n=5000
  ambiguity set: 16 measures

  replaying 163 events …
      err: time data "2026-05-11T14:50:12+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 38360. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
      err: time data "2026-05-07T09:28:06+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 5677. You might want to try:
    - passing `format` if your strings have a c

/Users/rithvikijju/edge-bot/kalshi_v2/backtest.py:281: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  post_q["received_at_utc"] = pd.to_datetime(post_q["received_at_utc"], utc=True)


      err: time data "2026-05-06T07:29:39+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 12344. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
    [20/163] KXBTCD-26MAY0605
    [40/163] KXBTCD-26MAY1105
    [60/163] KXBTCD-26MAY0716
    [80/163] KXBTCD-26MAY0710
      err: time data "2026-05-06T18:25:03+00:00" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z", at position 128. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might w

In [67]:
import pandas as pd
import math
from kalshi_v2.paper_db import _conn

conn = _conn()
df = pd.read_sql_query(
    "SELECT id, market_ticker, side, contracts, entry_price, "
    "settle_price, pnl_dollars, exit_reason, model_p_yes "
    "FROM trades WHERE trade_type = 'v2_live' AND settled = 1",
    conn)
conn.close()

# Apply the new accounting retroactively:
buffer = 0.02   # CFG['order_buffer_cents']/100
def fee(p):
    return math.ceil(0.07 * p * (1-p) * 100) / 100.0

# entry that the bot would have used post-fix: entry_recorded + buffer
# (the OLD entry_price stored is the pre-buffer ask, so add the buffer
# back to get the actual fill price)
df['entry_corrected'] = df['entry_price'] + buffer
# exit: for a TP win, the actual sell-side limit was yb - buffer where
# yb was such that mid moved +5c. Old settle_price was mid; the real
# exit fill was settle - half-spread - buffer ≈ settle - 0.04
df['exit_corrected']  = df['settle_price'].apply(lambda x: x - 0.04 if x > 0.5 else max(0.01, x))
df['fees_corrected']  = (df['entry_corrected'].apply(fee) + df['exit_corrected'].apply(fee)) * df['contracts']
df['pnl_corrected']   = (df['exit_corrected'] - df['entry_corrected']) * df['contracts'] - df['fees_corrected']

print(f"\n{'-'*70}")
print(f"  V2_LIVE TRADES — original accounting vs corrected")
print(f"{'-'*70}")
print(df[['id','market_ticker','side','contracts','entry_price','settle_price',
         'pnl_dollars','entry_corrected','exit_corrected','pnl_corrected']].to_string(index=False))

print(f"\n  Original WR:    {(df['pnl_dollars']>0).mean()*100:.1f}%  "
      f"(net {df['pnl_dollars'].sum():+.2f})")
print(f"  Corrected WR:   {(df['pnl_corrected']>0).mean()*100:.1f}%  "
      f"(net {df['pnl_corrected'].sum():+.2f})")


----------------------------------------------------------------------
  V2_LIVE TRADES — original accounting vs corrected
----------------------------------------------------------------------
 id          market_ticker side  contracts  entry_price  settle_price  pnl_dollars  entry_corrected  exit_corrected  pnl_corrected
 68 KXBTC-26MAY1112-B81250  yes         37         0.22         0.270        1.850             0.24           0.270  -3.700000e-01
 69 KXBTC-26MAY1112-B81250  yes         37         0.25         0.310        2.220             0.27           0.310  -6.661338e-16
 70 KXBTC-26MAY1112-B81250  yes         23         0.32         0.375        1.265             0.34           0.375  -1.150000e-01
 71 KXBTC-26MAY1112-B81350   no          9         0.80         0.635       -1.485             0.82           0.595  -2.385000e+00
 72 KXBTC-26MAY1112-B81350   no         11         0.64         0.605       -0.385             0.66           0.565  -1.485000e+00
 73 KXBTC-26MAY1112

## 12. Portfolio metrics

Paper view (CFG bankroll) and live view (Kalshi balance) are kept
separate. Open positions get marked-to-market via in-memory WS
books with REST fallback.

In [18]:
portfolio_metrics(kalshi_md=kalshi_md, kalshi_live=kalshi_live)

  PAPER PORTFOLIO
  cash:     $    100,000.00  (CFG['bankroll'])
  time:     2026-05-13T15:10:13.478939+00:00

  Cash:                $    101,700.84
  Open positions @ mark: $          0.00  (cost $0.00 + unreal $+0.00)
  Realized PnL:        $     +1,700.84
  TOTAL EQUITY:        $    101,700.84
    vs starting:       +1.70%

  Settled: n=82  wins=46 (56.1%)

  LIVE PORTFOLIO  (source: Kalshi API)
  time:     2026-05-13T15:10:13.654047+00:00

  Cash (spendable):      $        103.07
  Open positions @ mark: $          0.00  (cost $0.00 + unreal $+0.00)
  TOTAL EQUITY:          $        103.07



In [19]:
from kalshi_v2.paper_db import open_trades, settle_trade
from kalshi_v2.portfolio import _mark_to_market

op = open_trades()
if len(op) == 0:
    print('no open trades to resolve')
else:
    print(f'resolving {len(op)} open trades at current mark...\n')
    total_pnl = 0.0
    for _, t in op.iterrows():
        mark = _mark_to_market(t['market_ticker'], t['side'], kalshi_md=kalshi_md)
        if mark is None:
            mark = 0.5
        pnl = (mark - float(t['entry_price'])) * int(t['contracts'])
        settle_trade(int(t['id']), mark, pnl, 'manual_close')
        total_pnl += pnl
        print(f"  #{t['id']:<4} {t['market_ticker']:30s} {t['side']:>3s}  "
              f"x{t['contracts']:>3d} @ ${float(t['entry_price']):.3f}  "
              f"mark ${mark:.3f}  pnl ${pnl:+.2f}")
    print(f'\ntotal recorded pnl: ${total_pnl:+.2f}')

no open trades to resolve


## 13. Recent settled trades

In [68]:
st = settled_trades()
if len(st) == 0:
    print('no settled trades yet')
else:
    cols = ['id', 'market_ticker', 'side', 'contracts', 'entry_price',
            'settle_price', 'pnl_dollars', 'exit_reason', 'trade_type']
    cols = [c for c in cols if c in st.columns]
    print(f'{len(st)} settled trades, last 10:\n')
    print(st[cols].tail(10).to_string(index=False))
    print(f'\ntotal realized PnL: ${st["pnl_dollars"].sum():+.2f}')
    wins = (st['pnl_dollars'] > 0).sum()
    print(f'win rate: {wins}/{len(st)} = {wins/len(st)*100:.1f}%')

100 settled trades, last 10:

 id          market_ticker side  contracts  entry_price  settle_price  pnl_dollars              exit_reason trade_type
 91 KXBTC-26MAY1221-B80450  yes       6070         0.28         0.220     -364.200        stop_loss (-6.0c)         v2
 92 KXBTC-26MAY1221-B80650  yes       6078         0.24         0.290      303.900      take_profit (+5.0c)         v2
 93 KXBTC-26MAY1221-B80650  yes       6066         0.30         0.240     -363.960        stop_loss (-6.0c)         v2
 94 KXBTC-26MAY1221-B80650  yes       6074         0.26         0.310      303.700      take_profit (+5.0c)         v2
 95 KXBTC-26MAY1221-B80750  yes       6074         0.26         0.200     -364.440        stop_loss (-6.0c)         v2
 96 KXBTC-26MAY1221-B80650  yes       6068         0.31         0.365      333.740      take_profit (+5.5c)         v2
 97 KXBTC-26MAY1221-B80650  yes       6050         0.38         0.000    -2299.000     resolution:result=no         v2
 98 KXBTC-26MAY131

## 13b. Full trade log

Every entry, every robust decision, every settlement, in chronological
order. Re-runnable any time — pulls live from the SQLite DB.

In [69]:
from kalshi_v2.paper_db import _conn
from kalshi_v2.data import fmt_local
import pandas as pd

conn = _conn()
tr = pd.read_sql_query('SELECT * FROM trades ORDER BY timestamp_utc', conn)
rd = pd.read_sql_query('SELECT * FROM robust_decisions ORDER BY ts', conn)
conn.close()

print('=' * 78)
print(f'  TRADE LOG  ·  {len(tr)} trades  ·  {len(rd)} robust filter decisions')
print('=' * 78)

if len(tr) == 0 and len(rd) == 0:
    print('\n  no activity logged yet. The bot is waiting for a signal that')
    print('  clears the edge filter AND the robust ambiguity-set check.')
else:
    if len(rd):
        print(f'\nROBUST FILTER — {len(rd)} decisions:')
        for _, r in rd.iterrows():
            pf = '✓ PASS' if r['passed'] else '✗ REJECT'
            print(f"  {fmt_local(r['ts'])}  {pf}  {r['ticker']:32s} {r['side']:>3s}  "
                  f"entry=${r['entry_price']:.3f}  pass_rate={r['pass_rate']:.2f}  "
                  f"mean_edge={r['mean_edge_c']:+.1f}c  range=[{r['min_edge_c']:+.1f}, {r['max_edge_c']:+.1f}]")
    if len(tr):
        print(f'\nENTRIES — {len(tr)} trades:')
        for _, t in tr.iterrows():
            settled_flag = '· settled' if t['settled'] else '· open'
            ts = t['timestamp_utc'][:19] if t['timestamp_utc'] else '?'
            print(f"  #{t['id']:<3} {ts}  {t['market_ticker']:32s} {t['side']:>3s} "
                  f"x{t['contracts']} @ ${t['entry_price']:.3f}  "
                  f"edge={t['entry_edge_cents']:+.1f}c  ({t['trade_type']}) {settled_flag}")
        settled = tr[tr['settled'] == 1]
        if len(settled):
            print(f'\nSETTLEMENTS:')
            for _, t in settled.iterrows():
                print(f"  #{t['id']:<3} {fmt_local(t['settled_at']) if t['settled_at'] else '?'}  "
                      f"{t['market_ticker']:32s} settle=${t['settle_price']:.2f}  "
                      f"PnL=${t['pnl_dollars']:+.2f}  reason={t['exit_reason']}")
            wins = (settled['pnl_dollars'] > 0).sum()
            losses = (settled['pnl_dollars'] < 0).sum()
            print(f'\n  total realized PnL: ${settled["pnl_dollars"].sum():+.2f}')
            print(f'  win/loss/even:      {wins} / {losses} / {len(settled)-wins-losses}')
            if len(settled) > 0:
                print(f'  win rate:           {wins/len(settled)*100:.1f}%')

  TRADE LOG  ·  101 trades  ·  915 robust filter decisions

ROBUST FILTER — 915 decisions:
  14:59:47 CDT  ✗ REJECT  KXBTCD-26MAY0817-T79999.99        no  entry=$0.210  pass_rate=0.94  mean_edge=+4.5c  range=[-1.3, +10.9]
  16:24:27 CDT  ✓ PASS  KXBTC-26MAY0818-B80150            no  entry=$0.690  pass_rate=0.94  mean_edge=+3.6c  range=[-0.7, +6.2]
  16:24:32 CDT  ✓ PASS  KXBTC-26MAY0818-B80150            no  entry=$0.690  pass_rate=0.94  mean_edge=+3.6c  range=[-0.7, +6.2]
  16:24:38 CDT  ✓ PASS  KXBTC-26MAY0818-B80150            no  entry=$0.690  pass_rate=0.94  mean_edge=+3.0c  range=[-1.8, +5.8]
  16:24:43 CDT  ✓ PASS  KXBTC-26MAY0818-B80150            no  entry=$0.690  pass_rate=0.94  mean_edge=+3.0c  range=[-1.8, +5.8]
  16:24:48 CDT  ✓ PASS  KXBTC-26MAY0818-B80150            no  entry=$0.690  pass_rate=0.94  mean_edge=+3.0c  range=[-1.8, +5.8]
  16:24:54 CDT  ✓ PASS  KXBTC-26MAY0818-B80150            no  entry=$0.690  pass_rate=0.94  mean_edge=+2.9c  range=[-2.0, +5.8]
  16:30:19

## 14. Risk-block log

Last 20 signals that were rejected by `risk_preflight`. Each entry
shows the reasons (ticker dedup, exposure cap, balance floor, etc.).

In [23]:
if not RISK_BLOCKS:
    print('no risk blocks recorded')
else:
    for rb in RISK_BLOCKS[-20:]:
        print(f'  {rb["ts"][:19]}  {rb["ticker"]:30s} {rb["side"]:>3s}  '
              f'{rb["strategy"]}  → {"; ".join(rb["reasons"])}')

no risk blocks recorded


## 15. Controls

Run any of these as needed.

In [41]:
# Stop the decision loop. Threads exit at next sleep wake (~250 ms).
# stop_bot()

# Hard kill: stop + force paper mode + refresh sessions.
# kill_switch()

# Flip mode to live. Refuses if balance unknown or $0.
enable_live()


# Flip back to paper.
# disable_live()

# Cancel every resting Kalshi order.
# cancel_all_live_orders()

✓ LIVE TRADING ENABLED. balance=$103.07


In [58]:
import duckdb
from pathlib import Path

DB_PATH = Path("/Users/rithvikijju/Downloads/kalshi_ws_hourly_plus_btc15m_20260512/data/live_capture_gapless/live_capture_gapless_20260512_paused.duckdb").expanduser()   # ← edit this

con = duckdb.connect(str(DB_PATH), read_only=True)

# 1. Table list
print("=" * 70)
print(f"  DuckDB inspection: {DB_PATH}")
print("=" * 70)
tables = con.execute("SHOW TABLES").fetchdf()
print("\nTABLES:")
print(tables.to_string(index=False))

# 2. For each table: column schema + row count + sample
for tbl in tables.iloc[:, 0]:
    print("\n" + "─" * 70)
    print(f"  {tbl}")
    print("─" * 70)
    try:
        cols = con.execute(f"DESCRIBE {tbl}").fetchdf()
        print("schema:")
        print(cols.to_string(index=False))
        n = con.execute(f"SELECT COUNT(*) AS n FROM {tbl}").fetchone()[0]
        print(f"\nrows: {n:,}")
        sample = con.execute(f"SELECT * FROM {tbl} LIMIT 3").fetchdf()
        print(f"\nfirst 3 rows:")
        print(sample.to_string(index=False))

        # If the table looks time-y, show date range
        for c in cols['column_name']:
            cl = c.lower()
            if any(k in cl for k in ('time', 'ts', 'date', 'minute', 'bucket')):
                try:
                    rng = con.execute(
                        f"SELECT MIN({c}) AS min, MAX({c}) AS max FROM {tbl}"
                    ).fetchone()
                    print(f"  {c}: {rng[0]}  →  {rng[1]}")
                except Exception:
                    pass
    except Exception as e:
        print(f"  error: {e}")
con.close()

  DuckDB inspection: /Users/rithvikijju/Downloads/kalshi_ws_hourly_plus_btc15m_20260512/data/live_capture_gapless/live_capture_gapless_20260512_paused.duckdb

TABLES:
                           name
             capture_health_all
            coinbase_ticker_all
                 event_coverage
             order_decision_all
           order_decision_dedup
                signal_scan_all
              signal_scan_dedup
                source_manifest
                 ws_control_all
               ws_lifecycle_all
         ws_orderbook_delta_all
ws_orderbook_snapshot_level_all
           ws_orderbook_top_all
         ws_orderbook_top_dedup
          ws_orderbook_top_gaps
           ws_private_event_all

──────────────────────────────────────────────────────────────────────
  capture_health_all
──────────────────────────────────────────────────────────────────────
schema:
        column_name column_type null  key default extra
     received_at_ns      BIGINT  YES None    None  None
    r

schema:
    column_name column_type null  key default extra
 received_at_ns      BIGINT  YES None    None  None
received_at_utc     VARCHAR  YES None    None  None
  market_ticker     VARCHAR  YES None    None  None
   event_ticker     VARCHAR  YES None    None  None
            sid      BIGINT  YES None    None  None
            seq      BIGINT  YES None    None  None
        yes_bid      DOUBLE  YES None    None  None
    yes_bid_qty      DOUBLE  YES None    None  None
        yes_ask      DOUBLE  YES None    None  None
    yes_ask_qty      DOUBLE  YES None    None  None
         no_bid      DOUBLE  YES None    None  None
     no_bid_qty      DOUBLE  YES None    None  None
         no_ask      DOUBLE  YES None    None  None
     no_ask_qty      DOUBLE  YES None    None  None
       btc_spot      DOUBLE  YES None    None  None
         source     VARCHAR  YES None    None  None
  capture_label     VARCHAR  YES None    None  None
   capture_path     VARCHAR  YES None    None  None

row

In [54]:
dashboard()

  V2 DASHBOARD @ 2026-05-13 10:23:44 CDT

A. CONNECTIVITY
   threads alive:  4/4
     ✓ v2_spot_poller
     ✓ v2_ws_listener
     ✓ v2_event_tracker
     ✓ v2_decision
   ws connected:   True  (mode=websocket, msgs=3254)
   ws subscribed:  KXBTC-26MAY1312

B. TRACKING
   spot:           $79,320.01
   causal sigma:   0.1124
   event:          KXBTC-26MAY1312
   closes:         11:00:00 CDT  (ttl +36.3 min)
   books in mem:   188
   sample books:
     KXBTC-26MAY1312-B71250              yes_bid=0.0  yes_ask=0.01  floor=71200
     KXBTC-26MAY1312-B71350              yes_bid=0.0  yes_ask=0.01  floor=71300
     KXBTC-26MAY1312-B71450              yes_bid=0.0  yes_ask=0.01  floor=71400

C. SCANNER (this snapshot, not the running worker)
   188 markets in event scope, 0 pass edge filters
     KXBTC-26MAY1312-B79350           yes entry=0.190 edge=+15.8c  · entry 0.190 < 0.2
     KXBTC-26MAY1312-B79550            no entry=0.870 edge=+6.6c  · entry 0.870 > 0.8
     KXBTC-26MAY1312-B80050        

In [55]:
stop_bot()

stopping bot... (threads daemon-exit at next sleep wake)


In [329]:
from kalshi_v2.robust import SIZER
print("Lipschitz history per mode:")
for key, hist in SIZER.history.items():
    print(f"  {key}: {len(hist)} entries")
    for p, s in hist[-10:]:
        print(f"    price={p:.3f}, size={s}")

Lipschitz history per mode:
  v2_paper: 50 entries
    price=0.310, size=6068
    price=0.310, size=6068
    price=0.310, size=6068
    price=0.310, size=6068
    price=0.310, size=6068
    price=0.310, size=6068
    price=0.310, size=6068
    price=0.350, size=6044
    price=0.380, size=6050
    price=0.380, size=6050


In [330]:
from kalshi_v2.config import CFG
print('Current "awareness" knobs:')
for k in ['brti_dampening','empirical_blend','market_shrink',
          'no_side_edge_surcharge_cents','no_near_strike_distance_usd',
          'robust_min_pass_rate','robust_min_mean_edge_c']:
    print(f'  {k:30s} = {CFG.get(k)}')

# Per-side and recent-window split
from kalshi_v2.paper_db import settled_trades
import pandas as pd
st = settled_trades()
live = st[st['trade_type']=='v2_live'].sort_values('timestamp_utc')
if len(live) >= 5:
    cutoff = len(live) // 2
    early = live.iloc[:cutoff]; late = live.iloc[cutoff:]
    for label, df in [('Early v2_live', early), ('Recent v2_live', late)]:
        wins = (df['pnl_dollars']>0).sum()
        print(f'\n{label} ({len(df)} trades):')
        print(f'  PnL:        ${df["pnl_dollars"].sum():+.2f}')
        print(f'  win rate:   {wins/len(df)*100:.1f}%')
        for side in ['yes','no']:
            sd = df[df['side']==side]
            if len(sd):
                w = (sd['pnl_dollars']>0).sum()
                print(f'  {side.upper():3s}: n={len(sd)}, PnL=${sd["pnl_dollars"].sum():+.2f}, WR={w/len(sd)*100:.1f}%')

Current "awareness" knobs:
  brti_dampening                 = 0.8
  empirical_blend                = 0.7
  market_shrink                  = 0.1
  no_side_edge_surcharge_cents   = 3.0
  no_near_strike_distance_usd    = 115
  robust_min_pass_rate           = 1.0
  robust_min_mean_edge_c         = 1.0

Early v2_live (7 trades):
  PnL:        $+4.03
  win rate:   57.1%
  YES: n=3, PnL=$+5.33, WR=100.0%
  NO : n=4, PnL=$-1.31, WR=25.0%

Recent v2_live (8 trades):
  PnL:        $+0.16
  win rate:   62.5%
  NO : n=8, PnL=$+0.16, WR=62.5%


In [40]:
stop_bot()

stopping bot... (threads daemon-exit at next sleep wake)


In [26]:
from kalshi_v2.paper_db import _conn
import pandas as pd

conn = _conn()
df = pd.read_sql_query("SELECT * FROM trades ORDER BY timestamp_utc", conn)
conn.close()

print("By trade_type:")
print(df.groupby('trade_type').agg(
    n=('id','count'),
    settled=('settled','sum'),
    pnl=('pnl_dollars','sum'),
    wins=('pnl_dollars', lambda x: (x>0).sum())
).round(2))

live = df[df['trade_type'] == 'v2_live']
settled = live[live['settled'] == 1]
if len(settled):
    wins = (settled['pnl_dollars'] > 0).sum()
    premium = (settled['entry_price'] * settled['contracts']).sum()
    print(f'\nv2_live: {len(settled)} settled')
    print(f'  realized PnL:    ${settled["pnl_dollars"].sum():+.2f}')
    print(f'  win rate:        {wins}/{len(settled)} = {wins/len(settled)*100:.1f}%')
    print(f'  return on prem:  {settled["pnl_dollars"].sum()/premium*100:+.2f}%')
    print(f'  mean per trade:  ${settled["pnl_dollars"].mean():+.3f}')

By trade_type:
             n  settled      pnl  wins
trade_type                            
v2          67       67  4606.70    38
v2_live     15       15     4.19     9

v2_live: 15 settled
  realized PnL:    $+4.19
  win rate:        9/15 = 60.0%
  return on prem:  +3.24%
  mean per trade:  $+0.279


In [61]:
stop_bot()

stopping bot... (threads daemon-exit at next sleep wake)


In [27]:
live_portfolio_metrics(kalshi_md=kalshi_md, kalshi_live=kalshi_live)

  LIVE PORTFOLIO  (source: Kalshi API)
  time:     2026-05-12T23:16:58.697707+00:00

  Cash (spendable):      $         93.54
  Open positions @ mark: $          0.00  (cost $0.00 + unreal $+0.00)
  TOTAL EQUITY:          $         93.54



{'cash': 93.54,
 'payout': 0.0,
 'open_mark': 0,
 'unrealized': 0,
 'equity': 93.54,
 'n_open': 0}

In [34]:
portfolio_metrics()

  PAPER PORTFOLIO
  cash:     $    100,000.00  (CFG['bankroll'])
  time:     2026-05-12T23:34:14.880654+00:00

  Cash:                $    104,940.00
  Open positions @ mark: $          0.00  (cost $0.00 + unreal $+0.00)
  Realized PnL:        $     +4,940.00
  TOTAL EQUITY:        $    104,940.00
    vs starting:       +4.94%

  Settled: n=68  wins=39 (57.4%)

  LIVE PORTFOLIO — no auth client, skipping


In [29]:
import json
b = kalshi_live.get_balance()
print(json.dumps(b, indent=2))

p = kalshi_live.get_positions()
print('\nPOSITIONS:')
print(json.dumps(p, indent=2))

{
  "balance": 9354,
  "portfolio_value": 0,
  "updated_ts": 1778627824
}

POSITIONS:
{
  "cursor": "",
  "event_positions": [],
  "market_positions": []
}


In [30]:
status()

  V2 BOT STATUS @ 2026-05-12 18:17:06 CDT
  mode:          paper
  live_enabled:  False
  running:       True
  iter:          20
  trades:        0
  live balance:  $93.54

  Threads: 4
    v2_spot_poller             alive=True
    v2_ws_listener             alive=True
    v2_event_tracker           alive=True
    v2_decision                alive=True

  WebSocket:
    mode:          websocket
    connected:     True
    subscribed:    KXBTC-26MAY1220
    msgs received: 292
    last msg:      18:17:05 CDT

  Tracked event:  KXBTC-26MAY1220
  Books in mem:   188

  Empirical bank: 5000
  Ambiguity set:  16 measures

  Log tail:
    [18:15:27 CDT] WS connected
    [18:15:28 CDT] tracking: KXBTC-26MAY1220 closes 2026-05-13 00:00:00+00:00
    [18:15:29 CDT] REST seed: 188 markets for KXBTC-26MAY1220
    [18:15:34 CDT] REST seed: 188 markets for KXBTC-26MAY1220
    [18:15:35 CDT] WS resubscribed: 188 tickers
    [18:15:35 CDT] WS sub confirmed sid=1
    [18:15:35 CDT] WS sub confirmed sid=

In [57]:
stop_bot()

stopping bot... (threads daemon-exit at next sleep wake)
